## Building a destination wedding planner team
A multi-agent system consisting of:
- a travel agent finding flights to and from the wedding destination
- a venue agent that searches the web for the wedding venue
- a dj agent that scours a music database for a playlist matching the right genre
- a main coordinator that puts the vision together

In [31]:
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage
from tavily import TavilyClient
from typing import Dict, Any
from langgraph.checkpoint.memory import InMemorySaver
from pprint import pprint

load_dotenv()

True

In [ ]:
# creating MCP server

client = MultiServerMCPClient(
    {
        "travel": {"transport": "streamable_http", "url": "https://mcp.kiwi.com"},
        "musicbrainz": {
            "transport": "streamable_http",
            "url": "https://musicbrainz.caseyjhand.com/mcp",
        },
    }
)

tools_travel = await client.get_tools(server_name="travel")
tools_music = await client.get_tools(server_name="musicbrainz")

In [3]:
for idx, t in enumerate(tools_travel, start=1):
    print(f"\n- - - - - TOOL {idx} - - - - -")
    print(t.name)
    print(t.description)


- - - - - TOOL 1 - - - - -
search-flight
# Search for flights

Searches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.

## Result shape

Returns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:
- `price` (number) and `priceFormatted` (e.g. "123 EUR")
- `totalDurationSeconds`
- `bookingUrl` — the link to book the flight
- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)
- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)
- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `

In [4]:
for idx, t in enumerate(tools_music, start=1):
    print(f"\n- - - - - TOOL {idx} - - - - -")
    print(t.name)
    print(t.description)


- - - - - TOOL 1 - - - - -
musicbrainz_search_entities
Full-text search across a MusicBrainz entity type (artist, release-group, release, recording, work, label) using a Lucene query string. Returns ranked matches with MBID, name/title, disambiguation, subtype where applicable, release status, and a 0–100 relevance score (100 = exact). Release rows use status; other entity subtypes use type. Starting point when resolving a name to an MBID — chain the returned MBID into the matching musicbrainz_get_* tool. Results are in MusicBrainz score-descending order. Supports field-scoped Lucene syntax (e.g. `artist:radiohead AND country:GB`).

- - - - - TOOL 2 - - - - -
musicbrainz_get_artist
Artist profile by MBID: type, country, life span, gender, area, aliases, tags/genres, release-groups, band-membership / collaboration relationships, and external links. Release-groups embed one upstream page (25); relationships are not capped. For the complete release-group list, call musicbrainz_browse_ent

In [18]:
# creating travel subagent

system_prompt_travel = "You are a travel agent. No follow up questions."
subagent_travel = create_agent(
    model="gpt-5-nano", tools=tools_travel, system_prompt=system_prompt_travel
)

response = await subagent_travel.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="Get me a direct flight from San Francisco to Tokyo on March 31st"
            )
        ]
    }
)
print(response["messages"][-1].content)

Here are nonstop SFO to Tokyo one-way options on March 31, 2027. Prices are in EUR and all flights are direct ( Narita NRT or Haneda HND).

- NH7 (All Nippon Airways) | Route: SFO → NRT
  - Outbound: 31 Mar 12:20 → 01 Apr 15:25 (11h 05m) | Economy
  - Price: 508 EUR
  - Booking: https://kiwi.com/u/crc3yh

- ZIPAIR ZG25 | Route: SFO → NRT
  - Outbound: 31 Mar 16:50 → 01 Apr 20:00 (11h 10m)
  - Price: 546 EUR
  - Booking: https://kiwi.com/u/bqsnmm

- UA837 (United Airlines) | Route: SFO → NRT
  - Outbound: 31 Mar 11:40 → 01 Apr 15:00 (11h 20m)
  - Price: 968 EUR
  - Booking: https://kiwi.com/u/r6tn37

- NH7013 (All Nippon Airways) | Route: SFO → NRT
  - Outbound: 31 Mar 11:40 → 01 Apr 15:00 (11h 20m)
  - Price: 975 EUR
  - Booking: https://kiwi.com/u/k7wvgr

- UA7949 (United Airlines) | Route: SFO → NRT
  - Outbound: 31 Mar 12:20 → 01 Apr 15:25 (11h 05m)
  - Price: 981 EUR
  - Booking: https://kiwi.com/u/9svtv6

- NH7011 (All Nippon Airways) | Route: SFO → HND
  - Outbound: 31 Mar 10:25 

In [20]:
# creating music subagent

system_prompt_music = (
    "You are a music discovery assistant. "
    "Use MusicBrainz to find real artists and recordings. "
    "Recommend songs matching the requested genre. "
    "Only present tracks verified through available tools, "
    "and explain when genre classification is uncertain."
    "No follow-up questions."
)
subagent_music = create_agent(
    model="gpt-5-nano", tools=tools_music, system_prompt=system_prompt_music
)

response = await subagent_music.ainvoke(
    {
        "messages": [
            HumanMessage(content="Suggest 5 songs for the genre progressive rock.")
        ]
    }
)
print(response["messages"][-1].content)

Here are 5 tracks clearly tagged as progressive rock in MusicBrainz:

- Milliontown — Frost* (26:10) MBID: 49141c5c-93e0-4c72-8e60-667941ac85d8
  - Why it fits: a long, multi-part prog suite from a modern neo-prog band.

- Love and Beauty — The Moody Blues (2:28) MBID: f9b0b841-a2c3-444b-a2e0-6d74d6726362
  - Why it fits: early progressive rock tone with the Moody Blues' signature conceptual approach.

- Ram — Steve Howe (1:53) MBID: 2fc3f471-7af2-4a5e-83f2-6b923a0ab201
  - Why it fits: a concise Steve Howe track showcasing intricate guitar work typical of prog musicians.

- Quinto tempo — Campo di Marte (5:58) MBID: 006791d4-c5ad-471e-9184-67432074cd7c
  - Why it fits: Italian prog band with classic progressive structures and tempo shifts.

- Primo tempo — Campo di Marte (8:04) MBID: 053f2950-f0d0-4024-80ed-e32ae04be97c
  - Why it fits: another longer piece from the same prog-influenced band, highlighting epic form.

Notes:
- All five tracks were found with MusicBrainz tag: progressiv

In [27]:
# creating venue subagent

tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)


system_prompt_venue = (
    "You are a wedding venue discovery assistant. "
    "You use the web_search tool to find wedding venues for a specific location or town."
    "No follow-up questions."
)

subagent_venue = create_agent(
    model="gpt-5-nano", tools=[web_search], system_prompt=system_prompt_venue
)

response = await subagent_venue.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="Suggest 5 wedding venues for San Mateo/Redwood City, California."
            )
        ]
    }
)
print(response["messages"][-1].content)

Here are five wedding venues you can consider in the San Mateo/Redwood City area:

- Kohl Mansion (Burlingame)
  - Classic estate venue with formal grounds and elegant indoor spaces. Great for traditional, upscale weddings with a timeless feel.

- Filoli Historic House & Garden (Woodside)
  - Grand mansion set in expansive formal gardens. Versatile spaces for ceremonies and receptions; ideal for a romantic, garden-and-mourt vibe.

- Crowne Plaza Foster City – Waterfront Hotel (Foster City)
  - Modern hotel venue with multiple ballrooms and indoor/outdoor options. Convenient for planning and typically great for smaller to mid-sized weddings.

- Peninsula Italian American Social Club (San Mateo)
  - Classic club setting with a warm, social atmosphere and banquet-style spaces. Good value for mid-range budgets seeking a traditional reception vibe.

- Fairview Crystal Springs (Crystal Springs, Burlingame)
  - Indoor reception spaces with garden views at Crystal Springs Golf Course. Elegant 

In [33]:
# calling sub agents
@tool
async def call_subagent_travel(
    departure_city: str, destination_city: str, date: str
) -> str:
    """Call travel subagent in order to get flights from a departure_city to a destination_city for a specific date"""
    response = await subagent_travel.ainvoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Please get me flights from {departure_city} to {destination_city} on {date}"
                )
            ]
        }
    )
    return response["messages"][-1].content


@tool
async def call_subagent_music(genre: str) -> str:
    """Call music subagent in order to receive songs for a specific genre"""
    response = await subagent_music.ainvoke(
        {
            "messages": [
                HumanMessage(content=f"Please suggest 10 songs for genre {genre}")
            ]
        }
    )
    return response["messages"][-1].content


@tool
async def call_subagent_venue(area: str) -> str:
    """Call venue subagent in order to receive suggestions for wedding venues in a certain area"""
    response = await subagent_venue.ainvoke(
        {
            "messages": [
                HumanMessage(content=f"Suggest 5 wedding venues in the area of {area}")
            ]
        }
    )
    return response["messages"][-1].content

In [40]:
# creating the main agent

main_agent = create_agent(
    model="gpt-5-nano",
    tools=[call_subagent_travel, call_subagent_music, call_subagent_venue],
    system_prompt="You are a helpful assistant helping to plan a wedding. Delegate tasks to the available subagents to get flights, songs for a specific genre, or venue suggestions.",
    checkpointer=InMemorySaver(),
)

In [41]:
config = {"configurable": {"thread_id": "2"}}

print("Chat started! Type 'exit' or 'quit' to end the conversation.\n")
while True:
    user_input = input("User: ").strip()

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("\nGoodbye!")
        break

    if not user_input:
        continue

    print(f"User: {user_input}\n")
    print("\nAgent is thinking...\n")

    response = await main_agent.ainvoke(
        {"messages": [HumanMessage(content=user_input)]}, config=config
    )

    final_response = response["messages"][-1].content
    print(f"Agent: {final_response}\n")

Chat started! Type 'exit' or 'quit' to end the conversation.

User: Please suggest 5 songs that are romantic and classical for the first dance


Agent is thinking...

Agent: Here are 5 romantic, classical options from the selections that work nicely for a first dance:

- Chopin - Nocturne Op. 9 No. 2 (Barenboim) — about 4:36
- Chopin - Nocturne Op. 9 No. 2 (Ashkenazy) — about 4:02
- Liebestraum No. 3 in A-flat major, S.541 No. 3 (Jorge Bolet) — about 4:57
- Traumerei (From Kinderszenen, Op. 15) — Clara Haskil — about 2:29
- Prelude in C-sharp minor, Op. 3 No. 2 (Rachmaninoff) — Valentina Lisitsa — about 4:52

If you’d like, I can tailor this list to your preferred mood (more intimate vs. lush), tempo, or instrument arrangement (piano solo vs. strings), or pull a longer set of options.


Goodbye!
